# Pipeline de Entrenamiento en Google Colab — Norma ERP (Ecuador)

Este cuaderno entrena y exporta los artefactos de Inteligencia Artificial de **Norma ERP** a partir de la extracción íntegra (100%) de los 4 cuerpos normativos oficiales (1,880 artículos/secciones):
1. **Constitución de la República del Ecuador (2008)** — 481 artículos/secciones.
2. **Código del Trabajo del Ecuador** — 714 artículos/secciones.
3. **Ley de Seguridad Social (IESS)** — 383 artículos/secciones.
4. **Manual Metodológico OIT — Indicadores de Trabajo Decente (ODS 8)** — 302 secciones.

---

## Archivos que debes subir a Google Colab (Panel izquierdo -> Archivos)
1. `retriever_evaluator_training_colab.jsonl` *(1,880 tripletas contrastivas: query + positive + hard_negative extraídas al 100% de los 4 PDFs)*
2. `fine_tuning_ecuador_ods8_qwen.jsonl` *(163 muestras ChatML multi-dominio con cálculos exactos Decimal.js)*

## Qué produce este cuaderno
- **Fase 1 (Obligatoria para el Buscador y Evaluador RAG):**
  - Entrena el **Modelo Buscador Bi-Encoder (`intfloat/multilingual-e5-base`)** con `MultipleNegativesRankingLoss` sobre los 1,895 artículos legales.
  - Entrena el **Modelo Evaluador Cross-Encoder (`BAAI/bge-reranker-v2-m3` / `mmarco-mMiniLMv2-L12-H384-v1`)** para re-ranking de precisión legal y detección de alucinaciones.
  - Genera y descarga automáticamente el archivo **`trained_retriever_index.json`**, listo para copiarse en `backend/src/modules/sustainability/data/trained_retriever_index.json`.
- **Fase 2 (Opcional si deseas usar un Modelo Grande Local Fine-Tuneado además de DeepSeek API):**
  - Ejecuta Fine-Tuning QLoRA (4-bit con Unsloth) sobre **`Qwen/Qwen2.5-14B-Instruct`** (o `Qwen/Qwen2.5-7B-Instruct` en GPU T4 gratuita).
  - Exporta el modelo cuantizado en formato **`.gguf` (`q4_k_m`)** para ejecutarlo localmente con **Ollama** o **LM Studio**.

In [ ]:
# ==============================================================================
# PASO 1: INSTALACIÓN DE DEPENDENCIAS PARA EL BUSCADOR Y EVALUADOR (FASE 1)
# ==============================================================================
!pip install -q sentence-transformers datasets accelerate torch scikit-learn

In [ ]:
# ==============================================================================
# PASO 2: VERIFICACIÓN DE LOS DATASETS SUBIDOS A COLAB
# ==============================================================================
import os
import json

RETRIEVER_DATASET_PATH = "retriever_evaluator_training_colab.jsonl"
SFT_DATASET_PATH = "fine_tuning_ecuador_ods8_qwen.jsonl"

if not os.path.exists(RETRIEVER_DATASET_PATH):
    raise FileNotFoundError(
        f"No se encontró {RETRIEVER_DATASET_PATH}. Súbelo desde docs/09-sostenibilidad-ods/ al panel de archivos de Colab."
    )

with open(RETRIEVER_DATASET_PATH, "r", encoding="utf-8") as f:
    retriever_rows = [json.loads(line) for line in f if line.strip()]

print(f"[OK] Dataset de Buscador/Evaluador cargado: {len(retriever_rows)} artículos legales completos.")
print("Ejemplo de registro:", json.dumps(retriever_rows[0], indent=2, ensure_ascii=False)[:500])

In [ ]:
# ==============================================================================
# PASO 3: ENTRENAMIENTO DEL MODELO BUSCADOR (BI-ENCODER RETRIEVER)
# Entrena el buscador semántico con los 1,895 artículos de los 4 PDFs oficiales
# y exporta trained_retriever_index.json para el backend de Norma ERP
# ==============================================================================
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

BI_ENCODER_MODEL_NAME = "intfloat/multilingual-e5-base"
OUTPUT_RETRIEVER_DIR = "./norma_erp_legal_retriever"

print(f"Cargando modelo base Bi-Encoder: {BI_ENCODER_MODEL_NAME}...")
retriever_model = SentenceTransformer(BI_ENCODER_MODEL_NAME)

# Construcción de pares contrastivos (query, positive_passage, hard_negative_passage)
train_examples = []
for row in retriever_rows:
    query_text = f"query: {row['query']}"
    pos_text = f"passage: {row['article_code']} - {row['positive']}"
    neg_text = f"passage: {row['hard_negative']}"
    train_examples.append(InputExample(texts=[query_text, pos_text, neg_text]))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model=retriever_model)

print(f"Iniciando entrenamiento del Modelo Buscador sobre {len(train_examples)} artículos...")
retriever_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=int(len(train_dataloader) * 0.1),
    output_path=OUTPUT_RETRIEVER_DIR,
    show_progress_bar=True,
)
print(f"[OK] Modelo Buscador guardado en: {OUTPUT_RETRIEVER_DIR}")

In [ ]:
# ==============================================================================
# PASO 4: EXPORTACIÓN DEL ÍNDICE ENTRENADO (trained_retriever_index.json)
# Este archivo se coloca en: backend/src/modules/sustainability/data/trained_retriever_index.json
# ==============================================================================
import numpy as np

print("Calculando embeddings densos normalizados para los 1,895 artículos...")
passages = [f"passage: {r['article_code']} - {r['positive']}" for r in retriever_rows]
embeddings = retriever_model.encode(
    passages,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

# Reducción a precisión compacta de 4 decimales para carga instantánea en Node.js
exported_articles = []
for idx, row in enumerate(retriever_rows):
    exported_articles.append({
        "id": idx,
        "articleCode": row["article_code"],
        "title": row.get("title", ""),
        "category": row.get("category", "LABOR_CODE"),
        "odsTarget": row.get("ods_target", "8.8"),
        "domain": row.get("domain", "LABOR_CODE"),
        "embedding": [round(float(x), 4) for x in embeddings[idx][:128]],
    })

index_payload = {
    "model": BI_ENCODER_MODEL_NAME,
    "totalArticles": len(exported_articles),
    "embeddingDim": 128,
    "articles": exported_articles,
}

INDEX_FILENAME = "trained_retriever_index.json"
with open(INDEX_FILENAME, "w", encoding="utf-8") as f:
    json.dump(index_payload, f, ensure_ascii=False)

print(f"[OK] Índice exportado exitosamente: {INDEX_FILENAME} ({len(exported_articles)} artículos indexados).")

try:
    from google.colab import files
    files.download(INDEX_FILENAME)
except Exception:
    print("Descarga manual disponible en el panel lateral de archivos de Colab.")

In [ ]:
# ==============================================================================
# PASO 5: ENTRENAMIENTO DEL MODELO EVALUADOR (CROSS-ENCODER RERANKER)
# Califica relevancia exacta [0.0 a 1.0] entre la pregunta del usuario y el artículo legal
# ==============================================================================
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

CROSS_ENCODER_MODEL_NAME = "unicamp-dl/mMiniLM-L6-v2-mmarco-v2"
OUTPUT_EVALUATOR_DIR = "./norma_erp_legal_evaluator"

print(f"Cargando Cross-Encoder Evaluador base: {CROSS_ENCODER_MODEL_NAME}...")
evaluator_model = CrossEncoder(CROSS_ENCODER_MODEL_NAME, num_labels=1)

ce_train_samples = []
for row in retriever_rows:
    # Par positivo (relevancia = 1.0)
    ce_train_samples.append(
        InputExample(texts=[row["query"], f"{row['article_code']}: {row['positive']}"], label=1.0)
    )
    # Par negativo difícil del mismo dominio (relevancia = 0.0)
    ce_train_samples.append(
        InputExample(texts=[row["query"], row["hard_negative"]], label=0.0)
    )

ce_dataloader = DataLoader(ce_train_samples, shuffle=True, batch_size=16)

print(f"Entrenando Modelo Evaluador Cross-Encoder con {len(ce_train_samples)} pares...")
evaluator_model.fit(
    train_dataloader=ce_dataloader,
    epochs=1,
    warmup_steps=int(len(ce_dataloader) * 0.1),
    output_path=OUTPUT_EVALUATOR_DIR,
    show_progress_bar=True,
)
print(f"[OK] Modelo Evaluador guardado en: {OUTPUT_EVALUATOR_DIR}")

---
## Fase 2: Fine-Tuning QLoRA del Modelo Generativo Local (Qwen 2.5 / DeepSeek Distill)

> **Nota:** Si en producción utilizas la API de **DeepSeek** (`AI_PROVIDER=deepseek` con `DEEPSEEK_API_KEY`), la Fase 1 ya es suficiente porque el Buscador y el Evaluador le entregan los artículos verificados a DeepSeek.
> Ejecuta esta **Fase 2** cuando desees entrenar tu propio LLM local (`.gguf`) para correrlo sin costo por token en **Ollama** (`AI_PROVIDER=local_ollama`) o **LM Studio** (`AI_PROVIDER=local_openai`).

In [ ]:
# ==============================================================================
# PASO 6: INSTALACIÓN DE UNSLOTH PARA FINE-TUNING QLORA (4-BIT)
# ==============================================================================
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# ==============================================================================
# PASO 7: FINE-TUNING SUPERVISADO (SFT) CON CHATML Y EXPORTACIÓN GGUF
# Puedes elegir:
# - "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" (Recomendado para GPU T4 gratuita de Colab)
# - "unsloth/Qwen2.5-14B-Instruct-bnb-4bit" (Para GPU A100 / L4)
# - "unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit"
# ==============================================================================
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

dataset = load_dataset("json", data_files=SFT_DATASET_PATH, split="train")

def format_chatml(batch):
    texts = []
    for msgs in batch["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_chatml, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs_norma_erp",
        report_to="none",
    ),
)

trainer.train()

# Exportación a formato GGUF cuantizado (q4_k_m) para Ollama / LM Studio
GGUF_DIR = "norma_erp_ecuador_gguf"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
print(f"[OK] Modelo GGUF exportado en: {GGUF_DIR}")